# ASIA AQ CS

#### Liam Thompson, Jun Zhang

In [ ]:
from melodies_monet import driver

In [ ]:
import xarray as xr, numpy as np, pandas as pd

def pair_track_model(model_file, obs_file, mapping, resample="60s",
                     obs_scale=None, fill_below=-9999.0):
    """Pair a CAM '1s_1pt' along-track model with DC8 obs (same flight/day!)."""
    obs_scale = obs_scale or {"NO2": 0.001}        # obs unit_scale like the yaml

    # model: ncol -> time, (time, lev)
    m = xr.open_dataset(model_file, decode_times=True).swap_dims({"ncol": "time"}).sortby("time")
    pmid  = m["PMID"].values                        # (time, lev) Pa
    mtime = pd.to_datetime(m["time"].values)

    # obs: resample, then MASK fills BEFORE anything else
    odf = xr.open_dataset(obs_file).to_dataframe().reset_index()
    odf["time"] = pd.to_datetime(odf["time"])
    for c in odf.select_dtypes("number").columns:
        odf.loc[odf[c] <= fill_below, c] = np.nan   # mask -999999 / -9999 sentinels
    odf = (odf.set_index("time").resample(resample).mean(numeric_only=True)
              .dropna(subset=["pressure_obs"]).reset_index())
    for v, s in obs_scale.items():
        if v in odf: odf[v] = odf[v] * s

    # nearest model track sample per obs time
    mi = mtime.values.astype("datetime64[ns]").astype("int64")
    oi = odf["time"].values.astype("datetime64[ns]").astype("int64")
    pos = np.clip(np.searchsorted(mi, oi), 1, len(mi) - 1)
    j = np.where((oi - mi[pos - 1]) <= (mi[pos] - oi), pos - 1, pos)

    # warn if the obs day isn't  covered by the model track
    gap = np.abs(mi[j] - oi).max() / 1e9
    if gap > 3600:
        print(f"WARNING: max obs-model time gap = {gap/3600:.1f} h "
              f"-- wrong obs file for this model day?")

    out = {"time": odf["time"].values,
           "latitude":  odf.get("latitude",  odf.get("lat")).values,
           "longitude": odf.get("longitude", odf.get("lon")).values,
           "altitude":  odf["altitude"].values,
           "pressure_obs": odf["pressure_obs"].values}

    for mod_v, obs_v in mapping.items():
        if mod_v not in m:
            continue
        da    = m[mod_v]
        scale = 1e9 if "mol/mol" in da.attrs.get("units", "").lower() else 1.0   # conv to  ppb
        prof  = da.values * scale
        mvals = np.full(len(odf), np.nan)
        for i, (jj, p) in enumerate(zip(j, odf["pressure_obs"].values)):
            order = np.argsort(pmid[jj])
            mvals[i] = np.interp(p, pmid[jj][order], prof[jj][order], left=np.nan, right=np.nan)
        out[obs_v]          = odf[obs_v].values if obs_v in odf else np.nan
        out[f"{obs_v}_mod"] = mvals
    return pd.DataFrame(out)

In [ ]:
df = pair_track_model(
    "/glade/derecho/scratch/jzhan166/f.e30beta01.FCnudged.GEMS01ne30x8.XNOx_PHL_anthro.02/run/f.e30beta01.FCnudged.GEMS01ne30x8.XNOx_PHL_anthro.02.cam.aircraft_asiaaqdc8_1s_1pt.2024-02-13-06525.nc",
    #"/glade/campaign/acom/acom-weather/emmons/ASIAAQ_obs/DC8/asiaaq-mrg10_dc8_20240213_RA_20260509.ict",   # per-flight obs
    "/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/asiaaq_cs_06082026/preprocessing/dc8_data/asiaaq_dc8_merge_all.nc",
    mapping={"O3": "O3", "NO2": "NO2", "T": "temperature"},
)

print(df.head()); print(df[["O3","O3_mod"]].describe())
print(df.head()); print(df[["temperature","temperature_mod"]].describe())

In [ ]:
import json, numpy as np, pandas as pd, xarray as xr

def save_paired_mm(df, mapping, obs_label, model_label, out_nc):
    """Write the track-paired DataFrame in MM's aircraft paired-file schema."""
    obs_vars   = list(mapping.values())     # ['O3','NO2','temperature']
    model_vars = list(mapping.keys())       # ['O3','NO2','T']
    obs_names  = set(obs_vars)
    n   = len(df)
    dims = ("time", "x")
    f64 = lambda a: (dims, np.asarray(a, float).reshape(n, 1))
    f32 = lambda a: (dims, np.asarray(a, np.float32).reshape(n, 1))

    ds = xr.Dataset()
    ds["latitude"]     = f64(df["latitude"])
    ds["longitude"]    = f64(df["longitude"])
    ds["pressure_obs"] = f32(df["pressure_obs"])
    ds["altitude"]     = f64(df["altitude"])

    for mod_v, obs_v in mapping.items():
        if obs_v in df:                              # obs column (plain obs name)
            ds[obs_v] = f64(df[obs_v])
        mcol = f"{mod_v}_new" if mod_v in obs_names else mod_v   # model col: _new on collision
        ds[mcol] = f32(df[f"{obs_v}_mod"])

    t = pd.to_datetime(df["time"].values)
    ds = ds.assign_coords(time=("time", t))
    t0 = t[0]
    ds["time"].encoding = {"units": f"minutes since {t0:%Y-%m-%d %H:%M:%S}",
                           "calendar": "proleptic_gregorian", "dtype": "int64"}

    meta = {"type": "aircraft", "radius_of_influence": None,
            "obs": obs_label, "model": model_label,
            "model_vars": model_vars, "obs_vars": obs_vars,
            "filename": f"{obs_label}_{model_label}.nc"}
    ds.attrs = {"title": "", "format": "NetCDF-4",
                "dict_json": json.dumps(meta, indent=4),
                "group_name": f"{obs_label}_{model_label}"}
    ds.to_netcdf(out_nc)
    print("wrote", out_nc)
    

In [ ]:
mapping = {"O3": "O3", "NO2": "NO2", "T": "temperature"}
df = pair_track_model(
    "/glade/derecho/scratch/jzhan166/f.e30beta01.FCnudged.GEMS01ne30x8.XNOx_PHL_anthro.02/run/f.e30beta01.FCnudged.GEMS01ne30x8.XNOx_PHL_anthro.02.cam.aircraft_asiaaqdc8_1s_1pt.2024-02-13-06525.nc",
    #"/glade/campaign/acom/acom-weather/emmons/ASIAAQ_obs/DC8/asiaaq-mrg10_dc8_20240213_RA_20260509.ict",   # per-flight obs
    "/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/asiaaq_cs_06082026/preprocessing/dc8_data/asiaaq_dc8_merge_all.nc",
    mapping={"O3": "O3", "NO2": "NO2", "T": "temperature"},
)
save_paired_mm(df, mapping,
               obs_label="dc8", model_label="cam-chem-se-era5",
               out_nc="./output_021324/0213_jz_asiaaq_dc8_cam-chem-se-era5.nc4")

In [ ]:
import xarray as xr

ds = xr.open_dataset(
    "/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/jun_zhang_cs_06122026/output_021324/0213_jz_asiaaq_dc8_cam-chem-se-era5.nc4"
)

print(ds.time)

# Number of time steps
print("n_times =", ds.sizes["time"])

# Total span
print("start =", ds.time.min().values)
print("end   =", ds.time.max().values)
print("duration =", ds.time.max().values - ds.time.min().values)

In [ ]:
an = driver.analysis()
an

an.control = '/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/jun_zhang_cs_06122026/control_jz_dc8_cesm.yaml'

an.read_control()

an.open_models()
an.open_obs()

#an.pair_data()
#an.save_analysis()

an.read_analysis()
an.plotting()
an.stats()